In [28]:
## build a instance in neo4j and download the credentials.
## KnowledgeGraphs/Neo4j-2be841c0-Created-2026-04-05.txt

with open('/Users/subham/Desktop/GENAI/genai/48_KnowledgeGraphs/Neo4j-bc40e8f3-Created-2026-04-05.txt', 'r') as file:
    content = file.readlines()

for line in content:
    if line.startswith("NEO4J_URI="):
        NEO4J_URI = line.split("=", 1)[1].strip()
    elif line.startswith("NEO4J_USERNAME="):
        NEO4J_USERNAME = line.split("=", 1)[1].strip()
    elif line.startswith("NEO4J_PASSWORD="):
        NEO4J_PASSWORD = line.split("=", 1)[1].strip()

In [29]:
import os
os.environ["NEO4J_URI"] = NEO4J_URI
os.environ["NEO4J_USERNAME"] = NEO4J_USERNAME
os.environ["NEO4J_PASSWORD"] = NEO4J_PASSWORD

In [30]:
from neo4j import GraphDatabase

# URI examples: "neo4j://localhost", "neo4j+s://xxx.databases.neo4j.io"
URI = NEO4J_URI
AUTH = (NEO4J_USERNAME, NEO4J_PASSWORD)


with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()
    print("Connection established.")

Connection established.


In [31]:
from langchain_neo4j import Neo4jGraph

graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database="bc40e8f3"  
)

In [32]:
moview_query="""
LOAD CSV WITH HEADERS FROM
'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' as row

MERGE(m:Movie{id:row.movieId})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)
FOREACH (director in split(row.director, '|') | 
    MERGE (p:Person {name:trim(director)})
    MERGE (p)-[:DIRECTED]->(m))
FOREACH (actor in split(row.actors, '|') | 
    MERGE (p:Person {name:trim(actor)})
    MERGE (p)-[:ACTED_IN]->(m))
FOREACH (genre in split(row.genres, '|') | 
    MERGE (g:Genre {name:trim(genre)})
    MERGE (m)-[:IN_GENRE]->(g))


"""
graph.query(moview_query)

[]

In [33]:
import os
from dotenv import load_dotenv
load_dotenv()
groq_api_key=os.getenv("GROQ_API_KEY_3")

from langchain_groq import ChatGroq
llm=ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-120b")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x11bd004d0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x11b07ade0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [34]:
print(graph.get_schema)

Node properties:
Movie {id: STRING, released: DATE, title: STRING, imdbRating: FLOAT}
Person {name: STRING}
Genre {name: STRING}
Relationship properties:

The relationships:
(:Movie)-[:IN_GENRE]->(:Genre)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:ACTED_IN]->(:Movie)


In [35]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

examples = [
    {
        "question": "How many artists are there?",
        "query": "MATCH (a:Person)-[:ACTED_IN]->(:Movie) RETURN count(DISTINCT a)",
    },
    {
        "question": "Which actors played in the movie Casino?",
        "query": "MATCH (m:Movie {{title: 'Casino'}})<-[:ACTED_IN]-(a:Person) RETURN a.name",
    },
    {
        "question": "How many movies has Tom Hanks acted in?",
        "query": "MATCH (a:Person {{name: 'Tom Hanks'}})-[:ACTED_IN]->(m:Movie) RETURN count(m)",
    },
    {
        "question": "List all the genres of the movie Schindler's List",
        "query": "MATCH (m:Movie {{title: 'Schindler\\'s List'}})-[:IN_GENRE]->(g:Genre) RETURN g.name",
    },
    {
        "question": "Which actors have worked in movies from both the comedy and action genres?",
        "query": "MATCH (a:Person)-[:ACTED_IN]->(:Movie)-[:IN_GENRE]->(g1:Genre), (a)-[:ACTED_IN]->(:Movie)-[:IN_GENRE]->(g2:Genre) WHERE g1.name = 'Comedy' AND g2.name = 'Action' RETURN DISTINCT a.name",
    },
]

example_prompt = PromptTemplate.from_template(
    "User input: {question}\nCypher query: {query}"
)

cypher_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix="""
You are a Neo4j expert.

Given the following schema:
{schema}

Generate a syntactically correct Cypher query.

Only return the query.
""",
    suffix="User input: {question}\nCypher query:",
    input_variables=["question", "schema"],
)

print(cypher_prompt.format(
    question="How many artists are there?",
    schema=graph.get_schema
))


You are a Neo4j expert.

Given the following schema:
Node properties:
Movie {id: STRING, released: DATE, title: STRING, imdbRating: FLOAT}
Person {name: STRING}
Genre {name: STRING}
Relationship properties:

The relationships:
(:Movie)-[:IN_GENRE]->(:Genre)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:ACTED_IN]->(:Movie)

Generate a syntactically correct Cypher query.

Only return the query.


User input: How many artists are there?
Cypher query: MATCH (a:Person)-[:ACTED_IN]->(:Movie) RETURN count(DISTINCT a)

User input: Which actors played in the movie Casino?
Cypher query: MATCH (m:Movie {title: 'Casino'})<-[:ACTED_IN]-(a:Person) RETURN a.name

User input: How many movies has Tom Hanks acted in?
Cypher query: MATCH (a:Person {name: 'Tom Hanks'})-[:ACTED_IN]->(m:Movie) RETURN count(m)

User input: List all the genres of the movie Schindler's List
Cypher query: MATCH (m:Movie {title: 'Schindler\'s List'})-[:IN_GENRE]->(g:Genre) RETURN g.name

User input: Which actors have worked in mo

In [36]:
from langchain_core.output_parsers import StrOutputParser

cypher_chain = (
    cypher_prompt   # 👈 YOUR CUSTOM PROMPT HERE
    | llm
    | StrOutputParser()
)


from langchain_core.prompts import ChatPromptTemplate

answer_prompt = ChatPromptTemplate.from_template("""
Given the result:
{context}

Answer the question:
{question}
""")

answer_chain = (
    answer_prompt
    | llm
    | StrOutputParser()
)

def run_query(question: str):
    schema = graph.get_schema   # ⚠️ property, not function

    # ✅ Step 1: Generate Cypher using YOUR prompt
    cypher_query = cypher_chain.invoke({
        "question": question,
        "schema": schema
    })

    print("\nGenerated Cypher:")
    print(cypher_query)

    # ✅ Step 2: Execute
    result = graph.query(cypher_query)

    print("\nRaw Result:")
    print(result)

    # ✅ Step 3: Final Answer
    answer = answer_chain.invoke({
        "context": result,
        "question": question
    })

    return answer

In [37]:
response = run_query("Who was the director of the movie Casino?")
print("\nFinal Answer:")
print(response)


Generated Cypher:
MATCH (d:Person)-[:DIRECTED]->(m:Movie {title: 'Casino'})
RETURN d.name;

Raw Result:
[{'d.name': 'Martin Scorsese'}]

Final Answer:
The director of the movie *Casino* was **Martin Scorsese**.


In [38]:
response = run_query("Which actors played in the movie Casino?")
print("\nFinal Answer:")
print(response)


Generated Cypher:
MATCH (m:Movie {title: 'Casino'})<-[:ACTED_IN]-(a:Person) RETURN a.name

Raw Result:
[{'a.name': 'Robert De Niro'}, {'a.name': 'Joe Pesci'}, {'a.name': 'Sharon Stone'}, {'a.name': 'James Woods'}]

Final Answer:
The actors who appeared in **Casino** are:

- Robert De Niro  
- Joe Pesci  
- Sharon Stone  
- James Woods


In [39]:
response = run_query("actors who acted in multiple movies")
print("\nFinal Answer:")
print(response)


Generated Cypher:
MATCH (a:Person)-[:ACTED_IN]->(m:Movie)
WITH a, count(m) AS movieCount
WHERE movieCount > 1
RETURN a.name AS actor, movieCount;

Raw Result:
[{'actor': 'Tim Allen', 'movieCount': 2}, {'actor': 'Tom Hanks', 'movieCount': 2}, {'actor': 'Robin Williams', 'movieCount': 2}, {'actor': 'Walter Matthau', 'movieCount': 2}, {'actor': 'Sophia Loren', 'movieCount': 2}, {'actor': 'Angela Bassett', 'movieCount': 3}, {'actor': 'Steve Martin', 'movieCount': 2}, {'actor': 'Al Pacino', 'movieCount': 2}, {'actor': 'Robert De Niro', 'movieCount': 3}, {'actor': 'Val Kilmer', 'movieCount': 2}, {'actor': 'Julia Ormond', 'movieCount': 3}, {'actor': 'Harrison Ford', 'movieCount': 2}, {'actor': 'Jonathan Taylor Thomas', 'movieCount': 2}, {'actor': 'Brad Renfro', 'movieCount': 2}, {'actor': 'Powers Boothe', 'movieCount': 2}, {'actor': 'Raymond J. Barry', 'movieCount': 2}, {'actor': 'Michael J. Fox', 'movieCount': 2}, {'actor': 'Michael Douglas', 'movieCount': 2}, {'actor': 'Annette Bening', 'm